# MAIN-1: Ensembl vs CAT Annotation Concordance Sankey

Hierarchical Sankey/alluvial diagram showing progressive concordance between
Ensembl (anchor-sequence projection) and CAT (graph-based projection) annotations.

**Levels:**
1. Gene presence: Present in both | Ensembl only | CAT only
2. RBH status: RBH found | No RBH (subset of 'both')
3. Transcript concordance: Full | Partial | None (subset of 'RBH found')
4. CDS integrity: Intact | Partial | Disrupted (subset of coding RBH)

**Input:** `intermediate_spreadsheets/sankey/` from workflow

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.sankey import Sankey
import numpy as np
from pathlib import Path

# Configuration
RESULTS_DIR = Path('../results')  # Adjust to your pipeline output directory
SANKEY_DIR = RESULTS_DIR / 'intermediate_spreadsheets' / 'sankey'
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(exist_ok=True)

# Colour scheme
COLORS = {
    'high_confidence': '#1b4f72',   # Dark blue
    'concordant': '#2e86c1',        # Medium blue
    'partial': '#85c1e9',           # Light blue
    'discordant': '#e74c3c',        # Red
    'ensembl_only': '#f39c12',      # Orange
    'cat_only': '#e67e22',          # Dark orange
    'background': '#ecf0f1',        # Light grey
}

In [ ]:
# Load pre-computed Sankey data
per_asm = pd.read_csv(SANKEY_DIR / 'sankey_per_assembly_flows.tsv', sep='\t')
flow_counts = pd.read_csv(SANKEY_DIR / 'sankey_flow_counts.tsv', sep='\t')

print(f"Loaded data for {len(per_asm)} assemblies")
print(f"\nAggregate flow counts:")
display(flow_counts.T)

In [ ]:
# Compute median flows for the Sankey diagram
medians = {
    'l1_both': per_asm['l1_both'].median(),
    'l1_ens_only': per_asm['l1_ensembl_only'].median(),
    'l1_cat_only': per_asm['l1_cat_only'].median(),
    'l2_pass': per_asm['l2_rbh_pass'].median(),
    'l2_fail': per_asm['l2_rbh_fail'].median(),
    'l3_full': per_asm['l3_tx_full'].median(),
    'l3_partial': per_asm['l3_tx_partial'].median(),
    'l3_none': per_asm['l3_tx_none'].median(),
    'l4_intact': per_asm['l4_cds_intact'].median(),
    'l4_partial': per_asm['l4_cds_partial'].median(),
    'l4_disrupted': per_asm['l4_cds_disrupted'].median(),
}

# Print headline concordance numbers
total = medians['l1_both'] + medians['l1_ens_only'] + medians['l1_cat_only']
print(f"Median gene presence concordance: {medians['l1_both']/total*100:.1f}%")
print(f"Median RBH pass rate: {per_asm['l2_pct_pass'].median():.1f}%")
print(f"Median transcript concordance: {per_asm['l3_pct_full'].median():.1f}%")
print(f"Median CDS integrity: {per_asm['l4_pct_intact'].median():.1f}%")

In [ ]:
# --- Agreement-focused Sankey ---
fig, ax = plt.subplots(figsize=(14, 8))

# Normalise to percentages of total
total = medians['l1_both'] + medians['l1_ens_only'] + medians['l1_cat_only']

# Level positions and widths
levels = ['Gene\nPresence', 'Locus\nOverlap (RBH)', 'Transcript\nConcordance', 'CDS\nIntegrity']
x_positions = [0, 1, 2, 3]

# Draw horizontal stacked bars at each level
bar_height = 0.6

# Level 1
l1_vals = [medians['l1_both'], medians['l1_ens_only'], medians['l1_cat_only']]
l1_pcts = [v/total*100 for v in l1_vals]
l1_colors = [COLORS['concordant'], COLORS['ensembl_only'], COLORS['cat_only']]

# Level 2
l2_vals = [medians['l2_pass'], medians['l2_fail']]
l2_pcts = [v/medians['l1_both']*100 if medians['l1_both'] > 0 else 0 for v in l2_vals]
l2_colors = [COLORS['concordant'], COLORS['discordant']]

# Level 3
l3_vals = [medians['l3_full'], medians['l3_partial'], medians['l3_none']]
l3_total = sum(l3_vals)
l3_pcts = [v/l3_total*100 if l3_total > 0 else 0 for v in l3_vals]
l3_colors = [COLORS['high_confidence'], COLORS['partial'], COLORS['discordant']]

# Level 4
l4_vals = [medians['l4_intact'], medians['l4_partial'], medians['l4_disrupted']]
l4_total = sum(l4_vals)
l4_pcts = [v/l4_total*100 if l4_total > 0 else 0 for v in l4_vals]
l4_colors = [COLORS['high_confidence'], COLORS['partial'], COLORS['discordant']]

all_levels = [
    (l1_pcts, l1_colors, ['Both', 'Ensembl only', 'CAT only'], l1_vals),
    (l2_pcts, l2_colors, ['RBH pass', 'No RBH'], l2_vals),
    (l3_pcts, l3_colors, ['Full match', 'Partial', 'No match'], l3_vals),
    (l4_pcts, l4_colors, ['Intact', 'Partial disruption', 'Disrupted'], l4_vals),
]

for i, (pcts, colors, labels, vals) in enumerate(all_levels):
    left = 0
    for j, (pct, color, label, val) in enumerate(zip(pcts, colors, labels, vals)):
        bar = ax.barh(3 - i, pct, left=left, height=bar_height, color=color,
                      edgecolor='white', linewidth=0.5)
        if pct > 5:  # Only label if big enough
            ax.text(left + pct/2, 3 - i, f'{label}\n{pct:.1f}%\n(n={int(val):,})',
                    ha='center', va='center', fontsize=7, fontweight='bold', color='white')
        left += pct

ax.set_yticks(range(4))
ax.set_yticklabels(list(reversed(levels)), fontsize=11, fontweight='bold')
ax.set_xlabel('Percentage of level denominator', fontsize=12)
ax.set_xlim(0, 105)
ax.set_title('Ensembl vs CAT Annotation Concordance\n(Median across assemblies)', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_main1_sankey.png', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figure_main1_sankey.pdf', bbox_inches='tight')
plt.show()
print(f"Saved to {OUTPUT_DIR / 'figure_main1_sankey.png'}")

In [ ]:
# --- Per-assembly distribution strip plot ---
fig, ax = plt.subplots(figsize=(10, 6))

pct_cols = ['l1_pct_both', 'l2_pct_pass', 'l3_pct_full', 'l4_pct_intact']
labels = ['Gene presence\n(both)', 'RBH locus\noverlap', 'Transcript\nconcordance', 'CDS\nintegrity']

for i, (col, label) in enumerate(zip(pct_cols, labels)):
    vals = per_asm[col].dropna()
    jitter = np.random.normal(0, 0.1, len(vals))
    ax.scatter(np.full(len(vals), i) + jitter, vals, alpha=0.3, s=8, color=COLORS['concordant'])
    # Median + IQR
    med = vals.median()
    q25, q75 = vals.quantile(0.25), vals.quantile(0.75)
    ax.plot([i-0.3, i+0.3], [med, med], color='black', linewidth=2)
    ax.plot([i, i], [q25, q75], color='black', linewidth=1.5)

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title('Per-assembly concordance at each Sankey level', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure_main1_per_assembly_distribution.png', dpi=300, bbox_inches='tight')
plt.show()